> **AI-generated.** An AI assistant drafted this page, and no maintainer has reviewed it yet. Please report errors on the issue tracker.

# Your data: values, records, and batches

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TARPS-group/prob-pipe/blob/main/docs/user_guide/04_values_and_records.ipynb)

Use this chapter when you need to pass data into a computation or extract values from a result.
Operations accept plain numbers and arrays, and their results carry labels and other term properties.
The examples show numeric values first, then records and data stored in pandas or xarray.
The last section covers batches, which hold many values of one kind along named levels.

In [1]:
# On Google Colab, install ProbPipe; elsewhere this cell does nothing.
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab's preinstalled PyMC 5 does not work with ArviZ 1.x, so the install upgrades PyMC.
    %pip install --quiet "probpipe-core[pymc] @ git+https://github.com/TARPS-group/prob-pipe.git@main"

In [2]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import xarray as xr

from probpipe import (
    DistributionBatch,
    Gamma,
    Normal,
    NumericArray,
    NumericArrayBatch,
    NumericRecordBatch,
    Opaque,
    Record,
    function,
    log_prob,
    mean,
    sample,
    workflow_run,
)

## Passing plain data and reading results

An operation can accept a plain array or number and wrap its numeric result as a `NumericArray`.
Here `log_prob` evaluates the log density of a standard normal distribution at the plain number 0.5:

In [3]:
theta = Normal("theta", 0.0, 1.0)
log_density = log_prob(theta, 0.5)

print("result:", repr(log_density))
print("log density of theta at 0.5:", round(float(log_density), 4))

result: NumericArray('log Normal(theta)', shape=(), dtype=float32)
  -1.0439385
log density of theta at 0.5: -1.0439


The result is a `NumericArray` with scalar shape `()`, which `float` converts to a Python number.
Its label `log Normal(theta)` names it in probability notation as the log density of the distribution `Normal(theta)`, as [Tracking and reproducing results](09_how_results_are_computed.ipynb#labels-and-notation) explains.
A function decorated with `@function` returns a `NumericArray` from a plain array in the same way, as [Propagating uncertainty through functions](03_functions_and_lifting.ipynb) shows.

## Numeric values

Construct a `NumericArray` yourself when you want to label numeric data before passing it to a computation.
It holds one array.
Its constructor takes the array first and the label as the keyword `label=`:

In [4]:
weights = NumericArray(jnp.array([0.2, 0.5, 0.3]), label="weights")

print("value:", repr(weights))
print("label of the value:", weights.label)
print("shape of the array:", weights.shape)
print("stored array:", weights.raw)

value: NumericArray('weights', shape=(3,), dtype=float32)
  [0.2 0.5 0.3]
label of the value: weights
shape of the array: (3,)
stored array: [0.2 0.5 0.3]


The repr names the label, the shape, and the dtype, and the values follow on the next line.
NumPy's printing formats the values, so `np.set_printoptions` sets their precision and how a long array is shortened.
`.raw` is the stored array, which is the form that NumPy, JAX, and plotting code expect.

A `NumericArray` stores its array in the form it was given.
A NumPy array stays a NumPy array, and `.raw` is that array with its dtype unchanged:

In [5]:
heights = NumericArray(np.array([1.71, 1.64, 1.8]), label="heights")

print("type of the stored array:", type(heights.raw).__name__)
print("dtype of the stored array:", heights.raw.dtype)

type of the stored array: ndarray
dtype of the stored array: float64


## Records

A `Record` holds several named fields, such as the measurements of one survey site.
We build one from a mapping of field names to values, and `label=` names it.
A nested mapping becomes a nested record:

In [6]:
site = Record(
    {"count": jnp.array([3, 5, 2]), "area": 1.5, "habitat": {"forest": 0.7, "wetland": 0.3}},
    label="site",
)

print("record:", repr(site))
print("number of fields:", len(site))
print("field keys:", site.keys())

record: NumericRecord('site')
  count             [3 5 2]
  area              1.5
  habitat/forest    0.7
  habitat/wetland   0.3
number of fields: 4
field keys: ('count', 'area', 'habitat/forest', 'habitat/wetland')


Every field of this record is numeric, so `Record` returns a `NumericRecord`, which is the record class whose fields are all numeric arrays.
The nested mapping `habitat` became a nested record with the fields `forest` and `wetland`.
Each field has a key, which is its path in the record: the field names from the top of the record down to the field, joined by `/`.
`site.keys()` lists the keys in order, and `len(site)` counts the four fields.
`habitat` is an interior node of the record, which groups the fields `habitat/forest` and `habitat/wetland`.

`Record.from_fields` takes the fields as keyword arguments.
A record constructed without `label=` derives its label from its top-level field names:

In [7]:
point = Record.from_fields(x=1.0, y=2.0)

print("record:", repr(point))
print("label of the record:", point.label)

record: NumericRecord('record(x,y)')
  x   1.
  y   2.
label of the record: record(x,y)


Indexing a record by a key returns a numeric field as a `NumericArray`.
A nested field is indexed by its path, and a nested record by the path of its group:

In [8]:
print("field count:", repr(site["count"]))
print("field habitat/forest:", repr(site["habitat/forest"]))
print("same field, through its group:", repr(site["habitat"]["forest"]))
print("is habitat/wetland a key:", "habitat/wetland" in site)

field count: NumericArray('count', shape=(3,), dtype=int32)
  [3 5 2]
field habitat/forest: NumericArray('habitat/forest', shape=(), dtype=float32)
  0.7
same field, through its group: NumericArray('forest', shape=(), dtype=float32)
  0.7
is habitat/wetland a key: True


`at_path` returns the node at any path, either a field or a nested record:

In [9]:
print("node at habitat:", repr(site.at_path("habitat")))

node at habitat: NumericRecord('habitat')
  forest    0.7
  wetland   0.3


A record also carries a **spec**, which declares the structure of its fields.
Inspect it when you need to check what a computation receives.
For a numeric record, the spec gives the shape of each field:

In [10]:
print("spec of the record:", site.spec)

spec of the record: NumericRecordSpec(count=(3,), area=(), habitat=NumericRecordSpec(forest=(), wetland=()))


`count` has shape `(3,)`, and the other fields are scalars.
[Propagating uncertainty through functions](03_functions_and_lifting.ipynb#specs) describes the spec of each kind of value.
A change to a record returns a new record and leaves the original as it was.
`replace` sets the values at existing paths, and `without` drops fields or nested records:

In [11]:
larger_site = site.replace({"area": 2.0})
counts_only = site.without("habitat", "area")

print("area of the new record:", float(larger_site["area"]))
print("area of the original record:", float(site["area"]))
print("fields left after without:", counts_only.keys())

area of the new record: 2.0
area of the original record: 1.5
fields left after without: ('count',)


## Getting raw values out

`.raw` is the stored data of any ProbPipe value.
On a record it returns a nested `dict` of the stored fields:

In [12]:
site.raw

{'count': Array([3, 5, 2], dtype=int32),
 'area': Array(1.5, dtype=float32, weak_type=True),
 'habitat': {'forest': Array(0.7, dtype=float32, weak_type=True),
  'wetland': Array(0.3, dtype=float32, weak_type=True)}}

A record's `.raw` is a nested `dict`, so indexing it by a top-level field name returns that field's stored array or a nested `dict`.
For a nested field, index each dictionary in turn, as in `site.raw["habitat"]["forest"]`.
Indexing the record first and then reading `.raw` on the field returns the same array:

In [13]:
print("raw count, from the record:", site.raw["count"])
print("raw count, from the field:", site["count"].raw)
print("raw habitat:", site.raw["habitat"])

raw count, from the record: [3 5 2]
raw count, from the field: [3 5 2]
raw habitat: {'forest': Array(0.7, dtype=float32, weak_type=True), 'wetland': Array(0.3, dtype=float32, weak_type=True)}


A numeric record also flattens into one vector, which suits optimizers and other code that works on a single array.
`to_vector` concatenates the fields in the order of `site.keys()`, and `vector_size` is the length of the vector:

In [14]:
print("flat vector of the record:", site.to_vector())
print("length of the vector:", site.vector_size)

flat vector of the record: [3.  5.  2.  1.5 0.7 0.3]
length of the vector: 6


The vector holds the three counts, then the area, then the two habitat shares.
The tutorials read results with `.raw` as well, as [tutorial 2](../tutorials/02_fitting.ipynb) does with the posterior quantiles of each parameter.

## Field names and labels

A field returned by indexing is labeled by its key.
A labeled value stored as a field also takes the field's key as its label, because the key is what names it within the record:

In [15]:
holder = Record({"w": weights}, label="holder")

print("label of site['habitat/forest']:", site["habitat/forest"].label)
print("label of holder['w']:", holder["w"].label)

label of site['habitat/forest']: habitat/forest
label of holder['w']: w


The field names and the label answer different questions: the label names the record, and the field names name its parts.
`rename` renames fields by their paths, and the record keeps its label:

In [16]:
renamed = site.rename({"area": "area_km2", "habitat/forest": "habitat/woodland"})

print("record after renaming fields:", repr(renamed))

record after renaming fields: NumericRecord('site')
  count              [3 5 2]
  area_km2           1.5
  habitat/woodland   0.7
  habitat/wetland    0.3


A function that returns a mapping returns a record whose fields are the keys of the mapping:

In [17]:
@function
def tally(counts: jax.Array) -> dict[str, jax.Array]:
    return {"total": jnp.sum(counts), "largest": jnp.max(counts)}


summary = tally(jnp.array([3, 5, 2]))

print("result of tally:", repr(summary))
print("total count:", int(summary["total"]))

result of tally: NumericRecord('tally(counts)')
  total     10
  largest   5
total count: 10


`total` and `largest` are the outputs of the result, and a later step indexes them by these names.

## Opaque values

A field can hold any Python value.
A string, a pandas `DataFrame` with text columns, or another non-numeric object becomes an opaque field:

In [18]:
plots = pd.DataFrame({"plot": ["north", "south"], "count": [3, 5]})
survey = Record({"count": jnp.array([3, 5]), "observer": "Ana", "plots": plots}, label="survey")

print("record:", repr(survey))
print("field observer:", repr(survey["observer"]))
print("observer:", survey["observer"].raw)
print("spec of the record:", survey.spec)

record: Record('survey', fields=('count', 'observer', 'plots'))
field observer: Opaque('observer', type=str)
observer: Ana
spec of the record: RecordSpec(count=(2,), observer=OpaqueSpec(type=str), plots=OpaqueSpec(type=DataFrame))


A record with a non-numeric field is a plain `Record`, since a `NumericRecord` holds numeric fields only.
Indexing an opaque field returns an `Opaque` value, whose spec records the Python type it holds, and its `.raw` is the object itself.
`to_vector` applies only to a `NumericRecord`, and `survey.to_numeric()` raises `TypeError`, naming the field `observer`.

An `Opaque` value can also be constructed on its own, for example to hold a fitted model from another library:

In [19]:
observer = Opaque("Ana", label="observer")

print("value:", repr(observer))
print("stored object:", observer.raw)

value: Opaque('observer', type=str)
stored object: Ana


## pandas and xarray fields

A pandas `Series` or an xarray `DataArray` of numbers is a numeric field.
The record stores it in its native form, so the index of a `Series` and the dimensions and coordinates of a `DataArray` are kept:

In [20]:
moose = pd.Series(
    [31.0, 44.0, 38.0],
    index=pd.Index([1967, 1968, 1969], name="year"),
    name="moose",
)
temperature = xr.DataArray(
    np.array([[-12.0, -9.5], [-14.0, -11.0]]),
    dims=("site", "month"),
    coords={"site": ["north", "south"], "month": ["jan", "feb"]},
)
census = Record({"moose": moose, "temperature": temperature}, label="census")

print("record:", repr(census))
print("field temperature:", repr(census["temperature"]))

record: NumericRecord('census')
  moose         [31. 44. 38.]
  temperature   [[-12.   -9.5]
                 [-14.  -11. ]]
field temperature: NumericArray('temperature', shape=(2, 2), dtype=float64)
  [[-12.   -9.5]
   [-14.  -11. ]]


The record is a `NumericRecord`, and indexing returns each field as a `NumericArray` with the shape of the stored data.
`.raw` is the stored object, with its index or coordinates:

In [21]:
census["moose"].raw

year
1967    31.0
1968    44.0
1969    38.0
Name: moose, dtype: float64

In [22]:
census["temperature"].raw.sel(site="south")

<xarray.DataArray (month: 2)> Size: 16B
array([-14., -11.])
Coordinates:
  * month    (month) <U3 24B 'jan' 'feb'
    site     <U5 20B 'south'

Code that expects a JAX array converts a field with `jnp.asarray`, which drops the index and the coordinates:

In [23]:
print("moose counts as a JAX array:", jnp.asarray(census["moose"]))

moose counts as a JAX array: [31. 44. 38.]


[Tutorial 5](../tutorials/05_updating.ipynb) passes counts indexed by year as an xarray `DataArray` to `condition_on` and selects the counts of a span of years by year.

## Batches

Every kind of term has a batch form, which holds several elements of that kind along named groups of axes called **levels**, as [Propagating uncertainty through functions](03_functions_and_lifting.ipynb) introduces.
`NumericArrayBatch(values, level_names, label=...)` builds a batch of arrays, and its levels group the leading axes of `values`.
Each level in this example contains one axis.
Here the heights of three people at each of two sites form a batch on the levels `site` and `person`:

In [24]:
heights = NumericArrayBatch(
    jnp.array([[1.71, 1.64, 1.80], [1.55, 1.62, 1.70]]),
    ("site", "person"),
    label="heights",
)

print("batch:", repr(heights))
print("spec of the batch:", heights.spec)

batch: NumericArrayBatch(
    'heights',
    levels={'site': 2, 'person': 3},
    element=NumericArraySpec(shape=(), dtype=float32),
)
  [[1.71 1.64 1.8 ]
   [1.55 1.62 1.7 ]]
spec of the batch: BatchSpec(NumericArraySpec(shape=(), dtype=float32), site=2, person=3)


Each element is one height, so the elements' spec declares a scalar, and the levels hold the six repetitions.
A batch of six heights is therefore a different term from one array of six heights: a function called on the batch evaluates its body once per height, and a function called on the array evaluates it once on all six.

Indexing by position selects along the levels in order, and `at_levels` selects by level name.
A selection that keeps a level returns a smaller batch, and a selection that fixes every level returns an element, labeled by its position:

In [25]:
print("the first site:", repr(heights[0]))
print("the second person at the first site:", repr(heights[0, 1]))
print("the second person at every site:", repr(heights.at_levels(person=1)))
print("shape of the stored array:", heights.raw.shape)

the first site: NumericArrayBatch(
    'heights[site=0]',
    levels={'person': 3},
    element=NumericArraySpec(shape=(), dtype=float32),
)
  [1.71 1.64 1.8 ]
the second person at the first site: NumericArray('heights[site=0, person=1]', shape=(), dtype=float32)
  1.64


the second person at every site:

 NumericArrayBatch(
    'heights[person=1]',
    levels={'site': 2},
    element=NumericArraySpec(shape=(), dtype=float32),
)
  [1.64 1.62]
shape of the stored array: (2, 3)


`.raw` is the stored array, whose leading axes are the levels.

A batch of records stores one column per field.
Indexing by position returns one record, and indexing by a field name returns that field's column as a batch:

In [26]:
sites = NumericRecordBatch(
    {"count": jnp.array([[3, 5, 2], [4, 1, 2]]), "area": jnp.array([1.5, 2.0])},
    "site",
    label="sites",
)

print("batch of records:", repr(sites))
print("the second site:", repr(sites[1]))
print("fields of the second site:", sites[1].raw)
print("the area column:", repr(sites["area"]))

batch of records: NumericRecordBatch('sites', levels={'site': 2})
  count   [[3 5 2]
           [4 1 2]]
  area    [1.5 2. ]


the second site: NumericRecord('sites[site=1]')
  count   [4 1 2]
  area    2.
fields of the second site: {'count': Array([4, 1, 2], dtype=int32), 'area': Array(2., dtype=float32)}
the area column: NumericArrayBatch('area', levels={'site': 2}, element=NumericArraySpec(shape=()))
  [1.5 2. ]


Most batches come from operations.
`sample` returns its draws as a batch on the level `sample`, and the draws of a joint distribution form a batch of records.
`workflow_run(seed=...)` makes the draws reproducible, as [Tracking and reproducing results](09_how_results_are_computed.ipynb) explains:

In [27]:
mu_and_sigma = (Normal("mu", 0.0, 1.0) & Gamma("sigma", 2.0, 1.0))

with workflow_run(seed=0):
    draws_of_theta = sample(theta, sample_shape=5)
    draws_of_product = sample(mu_and_sigma, sample_shape=4)

print("draws of theta:", repr(draws_of_theta))
print("draws of the product:", repr(draws_of_product))

draws of theta: NumericArrayBatch(
    'theta ~ Normal',
    levels={'sample': 5},
    element=NumericArraySpec(shape=(), dtype=float32, support=real),
)
  [-0.40515503 -1.366459   -0.9853808  -0.6123491   0.67858565]
draws of the product: NumericRecordBatch('(mu, sigma) ~ Normal·Gamma', levels={'sample': 4})
  mu      [-0.80297685  1.2865425  -1.1136918   0.6906381 ]
  sigma   [3.763184   1.2194252  1.5483675  0.88666284]


A batch of draws carries the label of one draw, as `theta ~ Normal`.
Distributions and functions have batch forms as well.
A `DistributionBatch` holds distributions whose draws share one event spec, and an operation on it returns a batch of results, one per distribution:

In [28]:
growth = DistributionBatch(
    [Normal("growth", 0.0, 1.0), Normal("growth", 1.0, 2.0)], "region", label="growth"
)

print("the second distribution:", repr(growth[1]))
print("mean of each distribution:", mean(growth).raw)

the second distribution: Normal('growth', loc=1.0, scale=2.0, label='growth[region=1]')
mean of each distribution: [0. 1.]


## See also

- [Values and records](../api/values.md): the API reference for `Record`, `NumericRecord`, `NumericArray`, `Opaque`, and their batches.
- [Input and output specs](../api/input_and_output_specs.md): the specs that declare the structure of a value.
- [Distributions](02_distributions.ipynb): the distributions whose draws are values and records.
- [Propagating uncertainty through functions](03_functions_and_lifting.ipynb): functions that take and return values, batches of inputs, and the specs that declare a function's inputs and output.
- [Tracking and reproducing results](09_how_results_are_computed.ipynb): the provenance that every value records.
- [Tutorial 2](../tutorials/02_fitting.ipynb) reads posterior quantiles with `.raw`, and [tutorial 5](../tutorials/05_updating.ipynb) uses an xarray `DataArray` of counts.